
# SKIRTOR torus opening angle: geometry controls IR silicate and FIR bump

The SKIRTOR clumpy torus geometry is controlled by the half-opening angle
(``agn_oa_skirtor``), which determines how much of the accretion disc the
dusty material covers. Smaller opening angles (more pole-on geometry,
~20–30°) produce a compact torus that exposes the hot inner disc; larger
angles (more flared, ~50–60°) create a covering geometry that obscures the
disc and reprocess more UV/optical photons into the mid-infrared.

This sweep shows the silicate absorption (9.7 μm, prominent at edge-on
inclinations) and the FIR graybody bump (which grows with covering as
more disc energy is absorbed and reradiated). Both effects depend on the
interplay between opening angle and inclination.

## References
.. [1] Stalevski, M., Fritz, J., Baes, M., et al. 2012, MNRAS, 420, 2756
   — SKIRTOR radiative-transfer torus models.
.. [2] Stalevski, M., Ricci, C., Ueda, Y., et al. 2016, MNRAS, 458, 2288
   — Torus covering factor and dual AGN components.


In [ ]:
import os

os.environ["TF_CPP_MIN_LOG_LEVEL"] = "2"  # suppress XLA/PjRt C++ INFO+WARNING logs

import warnings

import jax
import jax.numpy as jnp
import matplotlib as mpl
import matplotlib.pyplot as plt
import numpy as np

import tengri
from tengri.analysis.plotting import setup_style

setup_style()
warnings.filterwarnings("ignore", message=".*BakedInBackend.*")
warnings.filterwarnings("ignore", message=".*deprecated.*")

C_AA_PER_S = 2.998e18

SFH = {"type": "const", "all_params": tengri.FIXED, "log_total_mass": -10.0}
DUST = {"type": "two_component", "all_params": tengri.FIXED, "tau_diff": 0.0, "tau_bc": 0.0}

ssp = tengri.load_ssp()

oa_values = np.array([20.0, 30.0, 40.0, 50.0, 60.0])
norm = mpl.colors.Normalize(vmin=oa_values.min(), vmax=oa_values.max())
cmap = plt.get_cmap("viridis")

model = tengri.SEDModel.build(
    ssp,
    sfh=SFH,
    dust=DUST,
    agn={
        "disc": {"type": "multicolor", "all_params": tengri.FIXED},
        "torus": {"type": "skirtor", "all_params": tengri.FIXED},
        "all_params": tengri.FIXED,
        "log_lbol": 12.0,
        "lum_ratio": 1.0,
        "cos_inc": 0.5,
    },
    redshift=tengri.Fixed(0.0),
)
baseline = dict(model.spec.sample(jax.random.PRNGKey(0)))

fig, ax = plt.subplots(figsize=(7.0, 4.5))
for oa in oa_values:
    params = {**baseline, "agn_oa_skirtor": jnp.float64(oa)}
    out = model.predict(params)
    wave = np.asarray(model.wavelengths)
    nu_lnu = (C_AA_PER_S / wave) * np.asarray(out.rest_sed())
    ax.loglog(wave, nu_lnu, color=cmap(norm(oa)), lw=1.4)

cbar = fig.colorbar(
    mpl.cm.ScalarMappable(norm=norm, cmap=cmap),
    ax=ax,
    pad=0.01,
    label=r"Torus opening angle $\theta_{\rm oa}$ [deg]",
)

ax.set(
    xlim=(100, 1e6),
    xlabel=r"Rest-frame wavelength $\lambda$ [$\mathrm{\AA}$]",
    ylabel=r"$\nu L_\nu$  [erg s$^{-1}$]",
)
ax.grid(True, which="minor", alpha=0.2, linestyle=":")

fig.tight_layout()
plt.savefig("plot_torus_opening_angle_sweep.png", dpi=150, bbox_inches="tight")